# Actividad 2026 · Unidad 3

**Atribución:** esta actividad adapta material del [Deep Reinforcement Learning Course de Hugging Face](https://huggingface.co/learn/deep-rl-course). El contenido original pertenece a sus autores y colaboradores.

**Traducción:** texto en español elaborado con asistencia de IA a partir del material original. Puede contener errores; ante una duda, consultá el notebook fuente.

[Abrir esta unidad en Colab](https://colab.research.google.com/drive/1K9z4ESHwSK2Qlatv_ccnp3UhrZCG6sW9)

Adaptación del [notebook oficial](https://github.com/huggingface/deep-rl-class/blob/bcd3410dd101f4f4f85fc8c08569b306b72c96e9/notebooks/unit3/unit3.ipynb). Usa Gymnasium/ALE y RL Baselines3 Zoo para entrenar DQN en Space Invaders. No requiere publicar en Hugging Face.


# Unidad 3: Deep Q Learning en Atari 👾 con RL Baselines3 Zoo

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/thumbnail.jpg" alt="Portada de la unidad 3">

Vas a entrenar un agente DQN para jugar Space Invaders con [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo), una herramienta basada en [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/) que permite entrenar, evaluar y ajustar agentes.

Usamos la [implementación básica de DQN](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html), sin extensiones como Double DQN, Dueling DQN o memoria de experiencias priorizada.


### Entorno 🎮

- [ALE/SpaceInvaders-v5](https://gymnasium.farama.org/environments/atari/space_invaders/)

La [documentación](https://gymnasium.farama.org/environments/atari/space_invaders/#variants) compara las variantes de Space Invaders.

### Biblioteca 📚

- [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo)


## Objetivos 🏆

Al terminar, vas a entender cómo configurar RL Baselines3 Zoo, entrenar un agente DQN y cargar el modelo guardado para evaluarlo. La publicación en Hugging Face Hub queda fuera de esta adaptación.


## Material del curso de aprendizaje por refuerzos profundo de Hugging Face

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Ilustración del curso"/>


En el [curso gratuito de Hugging Face](https://huggingface.co/learn/deep-rl-course) se estudia aprendizaje por refuerzos profundo en teoría y práctica, se utilizan distintas bibliotecas y se entrenan agentes en varios entornos. Consultá el [programa](https://simoninithomas.github.io/deep-rl-course) y su [comunidad de Discord](https://discord.gg/ydHrjt3WP5).


## Requisitos previos 🏗️

Antes de empezar, leé la [unidad 3 sobre Deep Q Learning](https://huggingface.co/deep-rl-course/unit3/introduction).


Si encontrás un problema en el material original, podés [abrir un issue en su repositorio](https://github.com/huggingface/deep-rl-class/issues).

In [ ]:
%pip install -q rl_zoo3 'gymnasium[atari]' ale-py


In [ ]:
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)
env = gym.make("ALE/SpaceInvaders-v5", frameskip=1, repeat_action_probability=0.0)
obs, info = env.reset(seed=42)
print("Atari listo:", obs.shape, env.action_space)
env.close()


## Entrená un agente Deep Q Learning para jugar Space Invaders 👾

Con RL Baselines3 Zoo necesitamos dos pasos. Primero, crear `dqn.yml` con los hiperparámetros de entrenamiento. Este es un ejemplo de configuración:

```yaml
ALE/SpaceInvaders-v5:
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4
  policy: 'CnnPolicy'
  n_timesteps: !!float 1e6
  buffer_size: 100000
  learning_rate: !!float 1e-4
  batch_size: 32
  learning_starts: 100000
  target_update_interval: 1000
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01
  optimize_memory_usage: False
```


La configuración utiliza un preprocesamiento de Atari (reducción de cuadros, escala de grises y apilado de cuatro cuadros) y una política `CnnPolicy`, que procesa las imágenes con una red convolucional. `buffer_size` fija cuántas transiciones se guardan para reutilizarlas en el entrenamiento.

El ejemplo configura **un millón de pasos**. Entrenar una política útil puede tardar mucho más que la prueba corta de ejecución; Colab puede interrumpir sesiones largas. Consultá `!nvidia-smi` para conocer la GPU disponible.


Para experimentar con los hiperparámetros, prestá especial atención a `learning_rate`, `buffer_size` y `batch_size`. Antes de cambiarlos, consultá la [documentación de DQN](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#parameters).


El segundo paso es iniciar el entrenamiento y guardar el modelo en `logs/` 📁.

En el comando, `--algo` elige el algoritmo, `-f` la carpeta de resultados y `-c` el archivo de configuración.


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
!python -m rl_zoo3.train --algo ________ --env ALE/SpaceInvaders-v5  -f _________  -c _________
```


#### Solución

In [ ]:
%%writefile dqn.yml
ALE/SpaceInvaders-v5:
  env_kwargs:
    frameskip: 1
    repeat_action_probability: 0.0
  env_wrapper: stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4
  policy: CnnPolicy
  n_timesteps: 1000000
  buffer_size: 100000
  learning_rate: 0.0001
  batch_size: 32
  learning_starts: 100000
  target_update_interval: 1000
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01


In [ ]:
!python -m rl_zoo3.train --algo dqn  --env ALE/SpaceInvaders-v5 -f logs/ -c dqn.yml

## Evaluá el agente 👀

RL Baselines3 Zoo ofrece `enjoy.py` para cargar y probar el modelo guardado. Ejecutalo durante la cantidad de pasos indicada para observar su comportamiento.


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
!python -m rl_zoo3.enjoy  --algo dqn  --env ALE/SpaceInvaders-v5  --no-render  --n-timesteps _________  --folder logs/
```


#### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
!python -m rl_zoo3.enjoy  --algo dqn  --env ALE/SpaceInvaders-v5  --no-render  --n-timesteps 5000  --folder logs/

## Para discutir
El entrenamiento DQN de Atari puede tardar horas. Para comprobar la instalación y el flujo, reducí temporalmente `n_timesteps` y `learning_starts` en `dqn.yml`; luego restaurá los valores antes de evaluar el aprendizaje.
